In [246]:
import os
import re
from dotenv import load_dotenv
from ibm_watsonx_ai.foundation_models import ModelInference
from ibm_watsonx_ai.metanames import GenTextParamsMetaNames as GenParams

load_dotenv()

True

In [247]:
credentials = {
    "url": os.getenv("WATSONX_URL"),
    "apikey": os.getenv("WATSONX_APIKEY")
}
project_id = os.getenv("WATSONX_PROJECT_ID")

In [248]:
db_fetched_data = """
뒤집으려고 온몸을 비틀어대더니 드디어 옆으로 휙 돌기 성공!하지만 한쪽 팔이 자꾸 껴서 낑낑거리며 울어버린다. 너무 귀여움.오늘 밤수 1번밖에 안 해서 엄마 컨디션 최고, 고마워 아가.
"""

In [249]:
refined_data = db_fetched_data.strip().replace('\n', ' ')
raw_lines = re.split(r'\.(?=\s|$)', refined_data)
lines = [line.strip() + "." for line in raw_lines if line.strip()]

step1_insights = ""
for i, line in enumerate(lines):
    step1_insights += f"{i+1}. {line}\n"
step1_insights = step1_insights.strip()


In [250]:
extract_params = {
    GenParams.DECODING_METHOD: "greedy",
    GenParams.MIN_NEW_TOKENS: 1,
    GenParams.MAX_NEW_TOKENS: 250,
    # 💡 동일 단어나 기호의 무한 반복을 연산 단계에서 강력하게 억제합니다.
    GenParams.REPETITION_PENALTY: 1.2,
    # 💡 2번 분류 완료 후 온점(.) 폭포나 주석을 시작하려고 엔터를 치는 순간 즉시 강제 종료합니다.
    GenParams.STOP_SEQUENCES: ["\n.", "#", "Step", "주의사항"],
    # 💡 단어 선택의 변조 확률을 최소화하여 외국어 혼용과 오탈자를 완벽하게 예방합니다.
    GenParams.TEMPERATURE: 0.1,
    GenParams.TOP_P: 0.1
}

# 1단계 라벨 추출 모델을 검증된 최신 Mistral Small 모델로 변경
extractor_model = ModelInference(
    model_id="mistralai/mistral-small-3-1-24b-instruct-2503", # 💡 지원 모델 목록 중 최선의 대안입니다.
    credentials=credentials,
    params=extract_params,
    project_id=project_id
)


In [251]:
extract_prompt = f"""[Instruction] 
당신은 육아 기록 및 텍스트 분석 전문가입니다. 
주어진 [Data]의 각 문장을 순서대로 정밀 분석하여 아래 [Output Format] 양식에 맞춰 오직 정답 라벨 결과만 깨끗하게 출력하고 작업을 즉시 마무리하세요.

각 문장 분석 시 아래 규칙을 철저히 따르세요:
1. 원문 철자 일치 원칙: 핵심명사와 행동명사를 추출하여 채워 넣을 때는 절대로 새로운 한국어 철자를 임의로 조합하거나 변형하지 마십시오. 오직 제공된 [Data] 원본 문장 속에 표기된 단어의 글자 형태(예: 비틀어대더니 -> 비틀어대다, 껴서 -> 끼다/껴서)를 100% 그대로 유지하여 올바른 한국어 맞춤법으로만 라벨을 완성하세요.
2. 감정을 예측할 때 흐름을 확인 후 감정을 작성하세요.
3. 육아 범주 분류 항목에는 식사, 수면, 배변, 체온 중 문맥과 일치하는 단어 1개만 기재하되, 비어있을 때는 '없음'을 기재하세요.
4. 대한민국 표준 한국어만 사용하여 정확하게 작성하세요.

[Data]
{step1_insights}
[Data]
{step1_insights}

[Output Format]
1. [첫 번째 문장 원문 내용이 이곳에 들어갑니다]
- 핵심: 단어1, 단어2
- 행동: 단어1, 단어2
- 수치+단위: 36.7도, 16ml, 1번
- 예측 감정: 슬픔, 기특함
- 육아 범주 분류 (식사, 수면, 배변, 체온 중 선택) : 수면 5시간, 배변 2번

[Output]
"""

In [252]:
# 1단계 추출 모델 실행 및 결과 받아오기
extract_response = extractor_model.generate(prompt=extract_prompt)
results_list = extract_response.get('results', [])
first_result = next(iter(results_list)) if isinstance(results_list, list) else results_list
step2_keywords = first_result.get('generated_text', '').strip()

perfect_match_input = step2_keywords


In [253]:
print("\n=== 2단계: 주요 라벨 단어 추출 완료 ===")
print(step2_keywords)


=== 2단계: 주요 라벨 단어 추출 완료 ===
1. 뒤집으려고 온몸을 비틀어대다니 드디어 옆으로 휙 돌기 성공! 하지만 한쪽 팔이 자꾸 끼어서 낑낑거리면서 울어버렸다.
 - 핵심: 몸, 팔
 - 행동: 뒤집다, 비틀어대다, 돌다, 끼다, 울다
 - 수치+단위: 없음
 - 예측 감정: 기쁨, 걱정
 - 육아 범주 분류: 없음

2. 너무 귀엽다. 오늘 밤수 1번 밖에 안해서 엄마 컨디션 최고, 고마워 아가.
 - 핵심: 밤수, 엄마
 - 행동: 없다
 - 수치+단위: 1번
 - 예측 감정: 감사함, 행복감
 - 육아 범주 분류: 수면


In [254]:
creative_params = {
    # 💡 무작위 샘플링을 버리고, 가장 일관성 있고 정확한 결정론적 greedy 방식을 채택합니다.
    GenParams.DECODING_METHOD: "greedy",
    GenParams.MIN_NEW_TOKENS: 50,
    GenParams.MAX_NEW_TOKENS: 600,
    # 💡 동일한 어휘나 서식을 기계적으로 무한 복사하는 현상을 원천 방어합니다.
    GenParams.REPETITION_PENALTY: 1.2,
    # 💡 단어 선택 변조 확률을 최소화하여 뜬금없는 영단어(russi)와 어조 반말 꼬임을 소멸시킵니다.
    GenParams.TEMPERATURE: 0.1,
    GenParams.TOP_P: 0.1,
    # 💡 일기 작성을 완료한 후 가상으로 3번 소설을 쓰려고 줄바꿈하는 순간 API를 강제 종료합니다.
    GenParams.STOP_SEQUENCES: ["\n\n", "[END]"]
}

writer_model = ModelInference(
    model_id="meta-llama/llama-3-3-70b-instruct",
    credentials=credentials,
    params=creative_params,
    project_id=project_id
)


In [255]:
diary_prompt = f"""너는 인스타그램에 오늘 하루 아이의 성장 기록을 다정하고 따뜻하게 공유하는 대한민국 엄마이다.
제공된 [정제된 육아 데이터 블록]의 각 번호에는 실제 핵심명사, 행동명사, 수치, 예측 감정단어 라벨 정보만 들어있다.
입력된 라벨 단락의 전체 개수와 완벽하게 일치하도록, 주어진 라벨 단어의 의미적 조각들을 자연스럽게 융합하여 번호당 정확히 한 문장씩의 일기만 순서대로 작성해라.

[필수 작성 규칙]
1. 문장 개수 일치: 제공된 데이터 블록의 번호 단락을 순서대로 모두 처리하되, 한 단락당 정확히 1문장씩만 작성하여 입력 데이터의 총 개수와 출력되는 일기의 총 줄 수가 완벽하게 일치하도록 마감해라.
2. 자연스러운 구어체 재창조: 라벨의 단어들을 기계적으로 연결하지 말고, 한국인 엄마가 일상에서 친구나 독자에게 고백하듯 다정하고 세련된 존댓말 구어체 문장으로 전면 재창조해라.
3. 복합 감정 융합: 각 번호에 기록된 감정 라벨(예: 염려, 안도 등)의 심리 상태를 문장 전체의 어조에 부드럽게 녹여내어 글의 감성을 극대화해라.
4. 주어 반복 생략: 문장마다 '아기가', '엄마가' 같은 주어를 반복하여 사용하지 말고, 상황과 행동 중심으로 매끄럽고 간결하게 서술해라.
5. 고유 수치 보존: 라벨 블록에 명시된 고유한 수치 단위 기호는 원래 속해 있던 해당 번호 문장 속에 형태 그대로 명확하게 포함하여 작성해라.
6. 순수한 문장 출력: 순번 기호나 문장부호 이외의 특수 기호는 모두 제외하고 오직 깨끗한 한글 문장만 출력해라. 한 문장이 끝날 때마다 무조건 줄바꿈을 수행해라.
7. 완전 종결 기호 표기: 모든 번호의 일기 작성을 완벽하게 마친 바로 다음 줄에 무조건 [END] 라고만 출력하고 생성을 즉시 마무리해라.

[정제된 육아 데이터 블록]
{perfect_match_input}

[Diary]:"""


In [256]:
writer_response = writer_model.generate(prompt=diary_prompt)
writer_results = writer_response.get('results', [])
first_writer_result = next(iter(writer_results)) if isinstance(writer_results, list) else writer_results
raw_diary = first_writer_result.get('generated_text', '').strip()

In [257]:
if "[END]" in raw_diary:
    raw_diary = raw_diary.split("[END]")[0].strip()

fixed_diary = ""
for line in raw_diary.split('\n'):
    line = line.strip()
    if not line:
        continue
    
    # 새로운 번호(1., 2.)나 대괄호([1번])로 시작하는 정상적인 줄바꿈만 엔터를 유지합니다.
    if re.match(r'^\d+[\.\s\-~)]+|^\s*\[\d+', line):
        fixed_diary += "\n" + line
    else:
        # 문장 중간에 쪼개진 찌꺼기 줄바꿈은 앞 문장 뒤에 띄어쓰기로 이어 붙입니다.
        fixed_diary += " " + line

raw_diary = fixed_diary.strip()

# 2. 텍스트 정제 (한자 및 특수문자 제거)
cleaned_diary = re.sub(r'[\u4e00-\u9fff]', '', raw_diary) 
cleaned_diary = re.sub(r'[^가-힣a-zA-Z0-9\s\.,!\?]', '', cleaned_diary) # 'ml', '도' 기호 보존용
cleaned_diary = re.sub(r'^\d+[\.\s\-~)]+', '', cleaned_diary, flags=re.MULTILINE) # 시작 넘버링 제거

# 3. 모델이 출력한 줄바꿈(\n)을 우선 신뢰하여 분리
diary_lines = [line.strip() for line in cleaned_diary.split('\n') if line.strip()]

# 4. 각 라인별 재정제
full_print_lines = []
for line in diary_lines:
    line = re.sub(r'^\d+[\.\s\-~)]+', '', line).strip()
    if line:
        full_print_lines.append(line)

# 5. 안전한 바이트 단위 축소 알고리즘 (한글 깨짐 방지)
def truncate_by_bytes(text, max_bytes=230):
    text_bytes = text.encode('utf-8')
    if len(text_bytes) <= max_bytes:
        return text
    
    # 안전하게 지정된 바이트만큼 자른 후, 깨진 바이트 무시하고 디코딩
    truncated = text_bytes[:max_bytes - 3].decode('utf-8', errors='ignore')
    return truncated.strip() + "..."

final_lines = []
for line in full_print_lines:
    # UTF-8 기준 바이트 수 체크
    line_bytes = line.encode('utf-8')
    
    if len(line_bytes) > 230:
        # 안전하게 227바이트까지 자른 후 깨진 멀티바이트 찌꺼기는 무시하고 디코딩
        line = line_bytes[:227].decode('utf-8', errors='ignore').strip() + "..."
        
    final_lines.append(line)

# 최종 다이어리 텍스트 병합 결과물
final_diary = "\n".join(final_lines)

In [258]:
print("\n=== 1단계: 구조화된 요약 메모 추출 완료 ===")
print(step1_insights)


=== 1단계: 구조화된 요약 메모 추출 완료 ===
1. 뒤집으려고 온몸을 비틀어대더니 드디어 옆으로 휙 돌기 성공!하지만 한쪽 팔이 자꾸 껴서 낑낑거리며 울어버린다.
2. 너무 귀여움.오늘 밤수 1번밖에 안 해서 엄마 컨디션 최고, 고마워 아가.


In [259]:
print("\n=== 2단계: 주요 라벨 단어 추출 완료 ===")
print(step2_keywords)


=== 2단계: 주요 라벨 단어 추출 완료 ===
1. 뒤집으려고 온몸을 비틀어대다니 드디어 옆으로 휙 돌기 성공! 하지만 한쪽 팔이 자꾸 끼어서 낑낑거리면서 울어버렸다.
 - 핵심: 몸, 팔
 - 행동: 뒤집다, 비틀어대다, 돌다, 끼다, 울다
 - 수치+단위: 없음
 - 예측 감정: 기쁨, 걱정
 - 육아 범주 분류: 없음

2. 너무 귀엽다. 오늘 밤수 1번 밖에 안해서 엄마 컨디션 최고, 고마워 아가.
 - 핵심: 밤수, 엄마
 - 행동: 없다
 - 수치+단위: 1번
 - 예측 감정: 감사함, 행복감
 - 육아 범주 분류: 수면


In [260]:
print("\n=== 3단계: 최종 완성된 감성 일기 ===")
for idx, final_line in enumerate(final_lines):
    print(f"[{idx+1}번 일기]: {final_line}")
print(f"\n-> 최종 결과물 총 문장 수: {len(final_lines)}줄")


=== 3단계: 최종 완성된 감성 일기 ===
[1번 일기]: 드디어 첫 번째로 옆으로 잘 돌아보는데 한쪽 팔은 왜 이렇게 계속 끼는지 모르겠어요, 그래도 넘 기뻐요!
[2번 일기]: 우리 아가는 밤에 1번밖에 안깨우니까 정말 편안해요, 고맙다고 해야죠!

-> 최종 결과물 총 문장 수: 2줄
